# SOLO value-v1 (masked ya está entrenado — NO correr masked aquí)
**Antes de la celda 1**: sube al panel de archivos de Colab tu checkpoint local
`checkpoints/masked-v1/best_model.pt` (a la MISMA ruta: crea la carpeta `checkpoints/masked-v1/`).
Sin ese archivo el value entrenaría desde cero y no sería comparable.
Criterio del piloto: `loss_mse` < 0.6 (vs 0.78 estancado) sin romper la CE (~3.0).


In [ ]:
!git clone https://github.com/oscar2697/chess-transformer.git 2>/dev/null; true
%cd /content/chess-transformer
!git pull origin main && git log --oneline -1
!pip install python-chess torch matplotlib langchain-openai plotly -q
print('setup ok')


In [ ]:
from google.colab import userdata
import os
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_KEY')
os.environ['LLM_MODEL'] = 'moonshotai/kimi-k3'
from src.graph_agents.llm_agents import get_llm
print('backend:', get_llm()[0])  # esperado: nvidia


In [ ]:
# VERIFICA el checkpoint subido — si esto falla, NO sigas: sube best_model.pt primero
import pathlib
ck = pathlib.Path('checkpoints/masked-v1/best_model.pt')
assert ck.exists(), f'FALTA {ck}: sube tu checkpoint local antes de entrenar'
print('ckpt ok:', ck, f'{ck.stat().st_size/1e6:.0f} MB')


In [ ]:
# Datos (sesion fresca: regenerar el mismo dataset de 570k)
!mkdir -p data/raw
!wget -q https://database.nikonoel.fr/lichess_elite_2024-01.zip -O data/raw/e01.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-02.zip -O data/raw/e02.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-03.zip -O data/raw/e03.zip
!unzip -o -q data/raw/e01.zip -d data/raw/ && unzip -o -q data/raw/e02.zip -d data/raw/ && unzip -o -q data/raw/e03.zip -d data/raw/
from src.graph_agents.llm_agents import run_agent
r = run_agent('data_engineer', 'Preprocess 3 elite months',
              overrides={'pgn_path': 'data/raw', 'elo_threshold': 2000,
                         'max_positions': 600000},
              auto=True)
print(r['result'])


In [ ]:
# VALUE PILOTO (3 epocas). Mira loss_mse: objetivo < 0.6.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Value-head pilot',
              overrides={'epochs': 3, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'value-v1', 'seed': 42,
                         'resume': True, 'use_amp': True,
                         'value_weight': 3.0, 'value_lr_mult': 5.0,
                         'init_ckpt': 'checkpoints/masked-v1/best_model.pt'},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# VALUE FULL (10 epocas, continua del piloto via resume). Re-ejecutable si muere.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Value-head full',
              overrides={'epochs': 10, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'value-v1', 'seed': 42,
                         'resume': True, 'use_amp': True,
                         'value_weight': 3.0, 'value_lr_mult': 5.0,
                         'init_ckpt': 'checkpoints/masked-v1/best_model.pt'},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# DESCARGA FINAL (value-v1)
!tar -czf /content/value_artifacts.tar.gz -C /content/chess-transformer \
    checkpoints/value-v1 experiments/training_log_value-v1.jsonl \
    experiments/training_results_value-v1.json data/processed/stats.json 2>/dev/null; true
!ls -lh /content/value_artifacts.tar.gz
from google.colab import files
files.download('/content/value_artifacts.tar.gz')
# Ademas descarga manual: checkpoints/value-v1/best_model.pt
